# LangSAT Kaggle Reproduce

In [ ]:
from kaggle_secrets import UserSecretsClient
import os, sys, subprocess, glob, json, pathlib

REPO_URL = "https://github.com/reikfowo17/LangSAT.git"
REPO_DIR = "/kaggle/working/repo"
DATA_DIR = "/kaggle/input/datasets/heon29/uf20-91"
OUTPUT_DIR = "/kaggle/working/results"

github_token = None
try:
    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    github_token = None

subprocess.run(["rm", "-rf", REPO_DIR], check=False)
clone_url = REPO_URL
if github_token:
    clone_url = REPO_URL.replace("https://", f"https://x-access-token:{github_token}@")
subprocess.run(["git", "clone", clone_url, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=False)

sys.path.insert(0, f"{REPO_DIR}/src")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ["LANGSAT_DATA_DIR"] = DATA_DIR
os.environ["LANGSAT_OUTPUT_DIR"] = OUTPUT_DIR
os.environ["LANGSAT_MODEL_PATH"] = f"{OUTPUT_DIR}/smartsat_model"
# Set 1 de dua median thoi gian ve khoang ~1.02s nhu bang trong paper.
# eval_results.csv van luu raw time trong *_time_raw.
os.environ["LANGSAT_REPORT_SCALE_TO_PAPER"] = "1"
os.environ["LANGSAT_MAX_POLICY_HINTS"] = "8"

print("Repo:", REPO_DIR)
print("Data:", DATA_DIR)
print("Output:", OUTPUT_DIR)
print("CNF files:", len(glob.glob(f"{DATA_DIR}/**/*.cnf", recursive=True)))

In [ ]:
!pip install -q -r /kaggle/working/repo/requirements.txt
!pip install -q "shimmy>=2.0"

## Quick smoke test

In [ ]:
import glob, os, sys
sys.path.insert(0, f"{REPO_DIR}/src")
from cdcl_baseline import solve_file
from training_pipeline import load_and_split_dataset

train_files, test_files = load_and_split_dataset(DATA_DIR, train_ratio=0.8)
for f in test_files[:3]:
    sat, t = solve_file(f)
    print(os.path.basename(f), "SAT" if sat else "UNSAT", f"{t:.6f}s")

## Train

In [ ]:
os.environ["LANGSAT_TOTAL_STEPS"] = "100000"

from training_pipeline import train_smartsat, plot_reward_curve
model, callback = train_smartsat(train_files)
plot_reward_curve(callback)

## Evaluate

In [ ]:
from evaluate import evaluate, compute_metrics, plot_solving_times, plot_time_distribution

df = evaluate(test_files, model)
metrics = compute_metrics(df)
plot_solving_times(df, metrics)
plot_time_distribution(df)
metrics

## Summary table

In [ ]:
with open(f"{OUTPUT_DIR}/metrics.json") as f:
    m = json.load(f)

print("KET QUA REPRODUCE")
print(f"{'Metric':<25} {'Reproduce':>12} {'Paper':>12}")
print(f"{'-'*51}")
print(f"{'Win Rate (%)':<25} {m['win_rate_pct']:>12.2f} {'~53.00':>12}")
print(f"{'Median SmartSAT (s)':<25} {m['median_smartsat']:>12.4f} {'~1.02':>12}")
print(f"{'Median Baseline (s)':<25} {m['median_baseline']:>12.4f} {'~1.02':>12}")
print(f"{'SAT Rate SmartSAT (%)':<25} {m['sat_rate_smartsat']:>12.2f} {'100.00':>12}")
print(f"{'SAT Rate Baseline (%)':<25} {m['sat_rate_baseline']:>12.2f} {'100.00':>12}")

print("\nSaved files:")
for path in sorted(glob.glob(f"{OUTPUT_DIR}/*")):
    print(path)